In [1]:
import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

np.random.seed(42)

In [2]:
NUM_USERS = 500
START_DATE = datetime(2025, 1, 1)
END_DATE = datetime(2025, 6, 30)

user_ids = [f"user_{i}" for i in range(NUM_USERS)]

behaviour_types = [
    "stable",
    "night_impulse",
    "salary_cycler",
    "weekend_spender",
    "burst_buyer"
]

user_behaviour_map = {
    user: random.choice(behaviour_types)
    for user in user_ids
}

In [3]:
user_salary_day = {
    user: random.randint(1, 5)  # salary between 1st–5th
    for user in user_ids
}

In [4]:
merchant_categories = [
    "groceries",
    "entertainment",
    "food_delivery",
    "shopping",
    "travel",
    "electronics",
    "utilities"
]

In [6]:
transactions = []

for user in user_ids:
    current_date = START_DATE

    while current_date <= END_DATE:

        # Base number of transactions per day
        base_tx = np.random.poisson(1)

        behaviour = user_behaviour_map[user]

        # Weekend effect
        if current_date.weekday() >= 5:
            if behaviour == "weekend_spender":
                base_tx += np.random.poisson(2)

        # Salary effect
        if current_date.day <= 3:
            if behaviour == "salary_cycler":
                base_tx += np.random.poisson(3)

        for _ in range(base_tx):

            hour = np.random.randint(0, 24)

            # Night impulse
            if behaviour == "night_impulse" and np.random.rand() < 0.3:
                hour = np.random.choice([23, 0, 1, 2]) # Corrected line: choose from valid night hours

            amount = np.random.gamma(2, 50)

            category = random.choice(merchant_categories)

            transactions.append([
                user,
                current_date,
                hour,
                amount,
                category
            ])

        current_date += timedelta(days=1)

In [7]:
df = pd.DataFrame(
    transactions,
    columns=[
        "user_id",
        "transaction_date",
        "hour_of_day",
        "transaction_amount",
        "merchant_category"
    ]
)

In [8]:
df["transaction_timestamp"] = pd.to_datetime(
    df["transaction_date"]
) + pd.to_timedelta(df["hour_of_day"], unit="h")

df["day_of_week"] = df["transaction_timestamp"].dt.dayofweek
df["is_weekend"] = df["day_of_week"] >= 5
df["day_of_month"] = df["transaction_timestamp"].dt.day

In [9]:
df["is_night_purchase"] = df["hour_of_day"].isin([23,0,1,2,3])

In [11]:
import os

output_dir = "../backend/data/synthetic"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "transactions_raw.csv"), index=False)

In [12]:
from google.colab import files

files.download(os.path.join(output_dir, "transactions_raw.csv"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>